# 📊 Exercise Grading System: Agentic RAG (4-hour workshop)
## For instructors

---

### How to use
1. Run the **Setup** cell once.
2. Run the **grading function** cell once.
3. Paste the student's **.ipynb file path** or **GitHub URL** → run grading.
4. Results will be **appended automatically** to `4hr_scores.csv` and `4hr_scores.json`.
5. Change the path → run for the next student.

### Scoring rubric (10 points)

| Step | Score | Criteria |
|---------|:-----:|-------|
| 1. Chunk + Embed + Search | 3 | Pipeline works, Qdrant score is correct |
| 2. Agent + Custom Tool | 3 | Tool works, Agent can call it, docstring explanation is clear |
| 3. RAG Agent + Judge | 4 | RAG answers all 3 questions, includes LLM-as-Judge, explanation is clear |



## 📦 Setup (Run )

In [ ]:
%%time
!pip install -q google-genai nbformat requests langchain-text-splitters

import hashlib, os, json, random, re, csv, requests
from datetime import datetime
import nbformat
from google import genai

# Gemini API
try:
 from google.colab import userdata
 GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 GOOGLE_API_KEY = input('🔑 Gemini API Key: ')

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-2.5-pro'

SCORES_CSV = '4hr_scores.csv'
SCORES_JSON = '4hr_scores.json'

if not os.path.exists(SCORES_CSV):
 with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
 writer = csv.writer(f)
 writer.writerow(['Timestamp', 'Student Name', 'Student ID', 'Phone', 'LINE ID',
 'Step1', 'Step2', 'Step3', 'Total',
 'AI Suspected', 'Feedback', 'File Path'])
 print(f'📄 {SCORES_CSV} ')
else:
 with open(SCORES_CSV, 'r') as f:
 existing = sum(1 for _ in f) - 1
 print(f'📄 {SCORES_CSV} {existing} ')

if not os.path.exists(SCORES_JSON):
 with open(SCORES_JSON, 'w') as f:
 json.dump([], f)

print(f'✅ Setup | Model: {MODEL}')

## 🔧 (Run )

In [ ]:
def read_notebook(filepath_or_url):
 """ notebook local path GitHub URL"""
 if filepath_or_url.startswith('http'):
 # GitHub URL → raw content
 raw_url = filepath_or_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
 print(f'📥 : {raw_url}')
 resp = requests.get(raw_url)
 resp.raise_for_status()
 nb = nbformat.reads(resp.text, as_version=4)
 else:
 # Local file
 with open(filepath_or_url, 'r', encoding='utf-8') as f:
 nb = nbformat.read(f, as_version=4)
 
 info = {'student_name': '', 'student_id': '', 'phone': '', 'line_id': ''}
 full_content = ''
 
 for i, cell in enumerate(nb.cells):
 if cell.cell_type == 'code':
 for key, var in [('student_id','STUDENT_ID'), ('student_name','STUDENT_NAME'),
 ('phone','PHONE'), ('line_id','LINE_ID')]:
 m = re.search(rf"{var}\s*=\s*['\"]([^'\"]+)['\"]", cell.source)
 if m:
 info[key] = m.group(1)
 
 ct = cell.cell_type.upper()
 full_content += f'\n\n=== CELL {i} ({ct}) ===\n{cell.source}'
 if hasattr(cell, 'outputs'):
 for out in cell.outputs:
 if hasattr(out, 'text'):
 full_content += f'\n--- OUTPUT ---\n{out.text}'
 elif hasattr(out, 'data'):
 for mime, data in out.data.items():
 if 'text' in mime:
 txt = data if isinstance(data, str) else '\n'.join(data)
 full_content += f'\n--- OUTPUT ---\n{txt}'
 
 return info, full_content




---
## ▶️ ()

 `NOTEBOOK_PATH` **Run cell ** 

In [ ]:
# ===== path GitHub URL =====
NOTEBOOK_PATH = '' # local path GitHub URL
# local: '/content/drive/MyDrive/submissions/student_hw.ipynb'
# GitHub: 'https://github.com/user/repo/blob/main/homework.ipynb'

assert NOTEBOOK_PATH, '❌ path GitHub URL!'

# 1. notebook
info, content = read_notebook(NOTEBOOK_PATH)
print(f'👤 {info["student_name"]} ({info["student_id"]})')
print(f'📱 {info["phone"]} | 💬 {info["line_id"]}')

# 🔍 
already_graded = False
if os.path.exists(SCORES_JSON):
 with open(SCORES_JSON, 'r') as f:
 prev = json.load(f)
 for p in prev:
 if p.get('student_id') == info['student_id']:
 already_graded = True
 print(f'⚠️ {info["student_id"]} (: {p.get("total_score", "?")})')
 break

if already_graded:
 overwrite = input('🔄 ? (y/n): ').strip().lower()
 if overwrite != 'y':
 print('⏭️ ')
 raise SystemExit()
 prev = [p for p in prev if p.get('student_id') != info['student_id']]
 with open(SCORES_JSON, 'w', encoding='utf-8') as f:
 json.dump(prev, f, ensure_ascii=False, indent=2)
 rows = []
 with open(SCORES_CSV, 'r', encoding='utf-8') as f:
 rows = list(csv.reader(f))
 with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
 writer = csv.writer(f)
 for row in rows:
 if len(row) > 2 and row[2] == info['student_id']:
 continue
 writer.writerow(row)
 print(f'🗑️ ')

# 2. 
expected = generate_expected(info['student_id'])
print(f'🔑 : chunks={expected["num_chunks"]}, query="{expected["query"][:30]}..."')

# 3. Gemini
print(f'🤖 {MODEL}...')
grade = grade_with_gemini(info['student_id'], content, expected)

# 4. 
print(f'\n{"="*60}')
print(f'📊 : {info["student_name"]} ({info["student_id"]})')
print(f'{"="*60}')
print(f' 1 (Pipeline): {grade["step1_score"]}/3 — {grade["step1_feedback"]}')
print(f' 2 (Agent): {grade["step2_score"]}/3 — {grade["step2_feedback"]}')
print(f' 3 (RAG): {grade["step3_score"]}/4 — {grade["step3_feedback"]}')
print(f' {"─"*56}')
print(f' 🏆 : {grade["total_score"]}/10')
print(f' 💬 {grade["overall_feedback"]}')
if grade.get('is_ai_suspected'):
 print(f' ⚠️ AI: {grade["ai_reason"]}')

# 5. Append
append_result(info, grade, NOTEBOOK_PATH)

## 📊 View All Scores


In [ ]:
import csv

with open(SCORES_CSV, 'r', encoding='utf-8') as f:
 reader = csv.reader(f)
 header = next(reader)
 rows = list(reader)

print(f'📊 {len(rows)} \n')
print(f'{"#":>3} {"":<20} {"":<12} {"S1":>4} {"S2":>4} {"S3":>4} {"":>5} {"AI?":>4}')
print('=' * 65)

for idx, row in enumerate(rows, 1):
 name, sid = row[1], row[2]
 s1, s2, s3, total = row[5], row[6], row[7], row[8]
 ai = '⚠️' if row[9] == 'True' else '✅'
 print(f'{idx:>3} {name:<20} {sid:<12} {s1:>4} {s2:>4} {s3:>4} {total:>5} {ai:>4}')

if rows:
 scores = [float(r[8]) for r in rows]
 print(f'\n📈 : ={min(scores):.1f}, ={max(scores):.1f}, ={sum(scores)/len(scores):.1f}')

## 💾 Download Score Files


In [ ]:
try:
 from google.colab import files
 files.download(SCORES_CSV)
 print(f'✅ {SCORES_CSV}')
except:
 print(f'📄 : {os.path.abspath(SCORES_CSV)}')